# 03. Validator Repair Trace Analysis

This notebook extracts more information from the LangSmith traces by focusing on the moments where the `itinerary_validator` rejected an itinerary and routed the system back to the execution agent.

The goal is to answer:

- What did the validator ask the execution agent to adjust?
- Which tools did the execution agent use to repair the plan?
- How broad was each repair: one slot, one day, or many days?
- Did repairs reduce cost, resolve timing/route issues, or create mutation churn?
- How does this compare to the baseline trace shape?

Important counting rule: use `langsmith_runs.csv` filtered to `run_type == "tool"` for execution-level tool counts. The full `tool_calls.csv` recursively extracts serialized LangGraph states and therefore overcounts repeated historical messages.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import math
import re
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path('..').resolve()
TP_TRACE_PATH = ROOT / 'data' / 'traces' / 'travel_agent_full'
BL_TRACE_PATH = ROOT / 'data' / 'traces' / 'baseline'
LOCAL_TRACE_ROOT = ROOT / 'data' / 'travelplans' / 'LangSmithTraces'

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 200)

print('Travel Agent trace path:', TP_TRACE_PATH)
print('Baseline trace path:', BL_TRACE_PATH)


In [ ]:
def safe_loads(value: Any) -> Any:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (dict, list)):
        return value
    try:
        return json.loads(value)
    except Exception:
        return None


def canonical(value: Any) -> str:
    return json.dumps(value, sort_keys=True, ensure_ascii=False, default=str)


def output_content(outputs_json: Any) -> str:
    obj = safe_loads(outputs_json)
    if not isinstance(obj, dict):
        return '' if obj is None else str(obj)
    out = obj.get('output', obj)
    if isinstance(out, dict):
        content = out.get('content')
        if content is not None:
            return content if isinstance(content, str) else canonical(content)
    return canonical(out)


def parse_tool_args(inputs_json: Any) -> dict[str, Any]:
    obj = safe_loads(inputs_json)
    return obj if isinstance(obj, dict) else {}


def extract_cost_eur(text: str) -> float | None:
    if not text:
        return None
    match = re.search(r'Total estimated cost:\s*€\s*([0-9][0-9,.]*)', text)
    if not match:
        match = re.search(r'total cost[^€]{0,40}€\s*([0-9][0-9,.]*)', text, flags=re.I)
    if not match:
        return None
    raw = match.group(1).replace(',', '')
    try:
        return float(raw)
    except ValueError:
        return None


def issue_types(feedback: str) -> list[str]:
    text = feedback.lower()
    checks = [
        ('budget/cost', r'budget|cost|price|under|€'),
        ('flight/transport feasibility', r'flight|return route|layover|airport|departure|arrival|train'),
        ('route/timing feasibility', r'timing|transition|travel time|overlap|linear|buffer|route'),
        ('pacing/senior comfort', r'relaxed|senior|physically|peak heat|walking|heat'),
        ('evidence/availability', r'verify|confirmed|opening|availability|booking|tickets'),
        ('luggage/logistics', r'luggage|checkout|lockers|bags'),
        ('preference coverage', r'nightlife|food budget|street food|vegan|restaurant|interest'),
        ('contradiction/internal consistency', r'contradict|impossible|inconsistent|broken|confused'),
    ]
    found = [label for label, pattern in checks if re.search(pattern, text)]
    return found or ['other']


def compact_sequence(names: list[str]) -> str:
    if not names:
        return ''
    compact = []
    prev = None
    count = 0
    for name in names:
        if name == prev:
            count += 1
        else:
            if prev is not None:
                compact.append(f'{prev}x{count}' if count > 1 else prev)
            prev = name
            count = 1
    compact.append(f'{prev}x{count}' if count > 1 else prev)
    return ' -> '.join(compact)


def mutated_day(args: dict[str, Any]) -> int | None:
    day = args.get('day_index')
    try:
        return int(day)
    except Exception:
        return None


MUTATION_TOOLS = {'add_slot', 'insert_slot', 'delete_slot', 'add_day', 'remove_day'}
SEARCH_TOOLS = {'search_web', 'search_flights', 'search_hotels', 'search_restaurants', 'search_attractions'}
READONLY_CHECK_TOOLS = {'view_plan', 'cost_summary', 'check_route_timing', 'build_place_distance_graph'}
ERROR_RE = re.compile(r'\b(?:timeout|timed out|failed|error|exception|rate limit|unavailable|could not|unable)\b', re.I)
UNCERTAIN_RE = re.compile(r'\b(?:uncertain|unknown|not confirm|not confirmed|appears|seems|likely|may|might|fallback|if available|check before|verify)\b', re.I)
FRAGILE_DOMAINS = {'google.com', 'maps.google', 'instagram.com', 'facebook.com', 'tripadvisor.com', 'booking.com', 'expedia.com', 'agoda.com', 'turbopass.com'}


## 1. Load Trace Tables

The Travel Agent full export contains all LangSmith runs. We immediately derive `tp_tools` from `run_type == "tool"`, which is the clean source for actual tool executions.

In [ ]:
tp_manifest = pd.read_csv(TP_TRACE_PATH / 'manifest.csv')
tp_runs = pd.read_csv(TP_TRACE_PATH / 'langsmith_runs.csv')
bl_manifest = pd.read_csv(BL_TRACE_PATH / 'manifest.csv')
bl_local_tools = pd.read_csv(BL_TRACE_PATH / 'local_tool_calls.csv')

tp_runs['start_time_dt'] = pd.to_datetime(tp_runs['start_time'], errors='coerce', utc=True)
tp_runs['end_time_dt'] = pd.to_datetime(tp_runs['end_time'], errors='coerce', utc=True)

tp_tools = tp_runs[tp_runs['run_type'] == 'tool'].copy()
tp_tools['tool_args'] = tp_tools['inputs_json'].map(parse_tool_args)
tp_tools['args_norm'] = tp_tools['tool_args'].map(canonical)
tp_tools['output_text'] = tp_tools['outputs_json'].map(output_content)
tp_tools['tool_cost_total'] = tp_tools['output_text'].map(extract_cost_eur)
tp_tools['has_error_language'] = tp_tools['output_text'].str.contains(ERROR_RE, na=False)
tp_tools['has_uncertainty_language'] = tp_tools['output_text'].str.contains(UNCERTAIN_RE, na=False)

bl_tools = bl_local_tools[bl_local_tools['source'] == 'ai_tool_call_request'].copy()

summary = pd.DataFrame([
    {'system': 'travel_agent', 'threads': tp_manifest['thread_id'].nunique(), 'run_rows': len(tp_runs), 'execution_tool_calls': len(tp_tools)},
    {'system': 'baseline', 'threads': bl_manifest['thread_id'].nunique(), 'run_rows': np.nan, 'execution_tool_calls': len(bl_tools)},
])
display(summary)


## 2. Validator Attempts and Repair Intervals

A repair interval is the time window after a failed `itinerary_validator` output and before the next validator output in the same thread. Tool calls in that interval are interpreted as the execution agent's attempted fix.

In [ ]:
validator_rows = []
validator_runs = tp_runs[tp_runs['name'] == 'itinerary_validator'].copy()
for _, row in validator_runs.iterrows():
    out = safe_loads(row['outputs_json'])
    if not isinstance(out, dict) or 'validation_feedback' not in out:
        continue
    validator_rows.append({
        'thread_id': row['thread_id'],
        'validator_run_id': row['id'],
        'start_time': row['start_time'],
        'end_time': row['end_time'],
        'start_time_dt': row['start_time_dt'],
        'end_time_dt': row['end_time_dt'],
        'validation_attempt': int(out.get('validation_attempts') or 0),
        'validation_passed': bool(out.get('validation_passed')),
        'validation_feedback': out.get('validation_feedback') or '',
    })

validator_attempts_raw = pd.DataFrame(validator_rows)
# LangSmith includes mirrored wrapper rows with identical feedback. Keep the latest row per logical attempt.
validator_attempts = (
    validator_attempts_raw
    .sort_values(['thread_id', 'validation_attempt', 'start_time_dt'])
    .groupby(['thread_id', 'validation_attempt', 'validation_passed', 'validation_feedback'], as_index=False)
    .last()
    .sort_values(['thread_id', 'validation_attempt'])
    .reset_index(drop=True)
)
validator_attempts = validator_attempts.merge(
    tp_manifest[['thread_id', 'root_run_id', 'query', 'travelplan_title', 'validation_attempts']],
    on='thread_id',
    how='left',
    suffixes=('', '_final'),
)
validator_attempts['issue_types'] = validator_attempts['validation_feedback'].map(issue_types)
validator_attempts['issue_types_text'] = validator_attempts['issue_types'].map(lambda xs: ', '.join(xs))
validator_attempts['feedback_preview'] = validator_attempts['validation_feedback'].str.replace(r'\s+', ' ', regex=True).str.slice(0, 260)

repair_threads = tp_manifest[tp_manifest['validation_attempts'] > 1]
display(Markdown(f'Repair threads: **{len(repair_threads)}**; failed validator attempts: **{int((validator_attempts.validation_passed == False).sum())}**'))
display(validator_attempts[['travelplan_title', 'validation_attempt', 'validation_passed', 'issue_types_text', 'feedback_preview']])


In [ ]:
repair_interval_rows = []
for thread_id, group in validator_attempts.sort_values(['thread_id', 'validation_attempt']).groupby('thread_id'):
    group = group.reset_index(drop=True)
    for i, attempt in group.iterrows():
        if attempt['validation_passed']:
            continue
        next_start = group.loc[i + 1, 'start_time_dt'] if i + 1 < len(group) else pd.NaT
        mask = (
            (tp_tools['thread_id'] == thread_id)
            & (tp_tools['start_time_dt'] > attempt['end_time_dt'])
        )
        if pd.notna(next_start):
            mask &= tp_tools['start_time_dt'] < next_start
        tools = tp_tools[mask].sort_values('start_time_dt').copy()
        names = tools['name'].fillna('').tolist()
        mutation_tools = tools[tools['name'].isin(MUTATION_TOOLS)].copy()
        search_tools = tools[tools['name'].isin(SEARCH_TOOLS)].copy()
        check_tools = tools[tools['name'].isin(READONLY_CHECK_TOOLS)].copy()
        mutated_days = sorted({d for d in mutation_tools['tool_args'].map(mutated_day) if d is not None})
        cost_points = tools[tools['name'] == 'cost_summary']['tool_cost_total'].dropna().tolist()
        slot_count = int((tools['name'].isin({'add_slot', 'insert_slot', 'delete_slot'})).sum())
        repair_interval_rows.append({
            'thread_id': thread_id,
            'travelplan_title': attempt['travelplan_title'],
            'root_run_id': attempt['root_run_id'],
            'attempt': attempt['validation_attempt'],
            'next_validation_passed': bool(group.loc[i + 1, 'validation_passed']) if i + 1 < len(group) else None,
            'issue_types': attempt['issue_types_text'],
            'validator_feedback': attempt['validation_feedback'],
            'feedback_preview': attempt['feedback_preview'],
            'tool_calls': len(tools),
            'search_calls': len(search_tools),
            'mutation_calls': len(mutation_tools),
            'readonly_check_calls': len(check_tools),
            'slot_mutation_calls': slot_count,
            'distinct_days_mutated': len(mutated_days),
            'days_mutated': ', '.join(map(str, mutated_days)),
            'mutation_churn_score': round(slot_count / max(1, len(mutated_days)), 2),
            'cost_summary_calls': int((tools['name'] == 'cost_summary').sum()),
            'first_cost_summary': cost_points[0] if cost_points else np.nan,
            'last_cost_summary': cost_points[-1] if cost_points else np.nan,
            'cost_delta_within_repair': (cost_points[-1] - cost_points[0]) if len(cost_points) >= 2 else np.nan,
            'tool_sequence': ' -> '.join(names),
            'tool_sequence_compact': compact_sequence(names),
            'tool_counts': dict(Counter(names)),
        })

repair_intervals = pd.DataFrame(repair_interval_rows)
display(repair_intervals[[
    'travelplan_title', 'attempt', 'next_validation_passed', 'issue_types', 'tool_calls',
    'search_calls', 'mutation_calls', 'distinct_days_mutated', 'mutation_churn_score',
    'first_cost_summary', 'last_cost_summary', 'cost_delta_within_repair', 'tool_sequence_compact'
]])


## 3. Validator Feedback Taxonomy

This groups failed validator feedback into operational issue classes. One feedback message can belong to multiple classes.

In [ ]:
issue_rows = []
for _, row in repair_intervals.iterrows():
    for issue in row['issue_types'].split(', '):
        issue_rows.append({
            'travelplan_title': row['travelplan_title'],
            'attempt': row['attempt'],
            'issue_type': issue,
            'tool_calls': row['tool_calls'],
            'mutation_calls': row['mutation_calls'],
            'next_validation_passed': row['next_validation_passed'],
        })
issue_table = pd.DataFrame(issue_rows)
issue_summary = (
    issue_table.groupby('issue_type')
    .agg(
        failed_attempts=('attempt', 'size'),
        avg_tool_calls=('tool_calls', 'mean'),
        avg_mutation_calls=('mutation_calls', 'mean'),
        next_pass_rate=('next_validation_passed', 'mean'),
    )
    .reset_index()
    .sort_values(['failed_attempts', 'avg_mutation_calls'], ascending=False)
)
issue_summary['avg_tool_calls'] = issue_summary['avg_tool_calls'].round(1)
issue_summary['avg_mutation_calls'] = issue_summary['avg_mutation_calls'].round(1)
issue_summary['next_pass_rate'] = issue_summary['next_pass_rate'].round(2)
display(issue_summary)


## 4. Repair Locality and Mutation Churn

Repair locality asks whether one validator complaint causes edits to one day or many days. A high churn score means the execution agent repeatedly deleted/inserted slots per affected day.

In [ ]:
locality_cols = [
    'travelplan_title', 'attempt', 'issue_types', 'tool_calls', 'slot_mutation_calls',
    'distinct_days_mutated', 'days_mutated', 'mutation_churn_score', 'next_validation_passed'
]
repair_locality = repair_intervals[locality_cols].sort_values('mutation_churn_score', ascending=False)
display(repair_locality)

display(Markdown('### Mutation Tool Mix by Repair Attempt'))
mutation_mix_rows = []
for _, interval in repair_intervals.iterrows():
    start = validator_attempts[
        (validator_attempts.thread_id == interval.thread_id)
        & (validator_attempts.validation_attempt == interval.attempt)
        & (validator_attempts.validation_passed == False)
    ].iloc[0]
    next_attempt = validator_attempts[
        (validator_attempts.thread_id == interval.thread_id)
        & (validator_attempts.validation_attempt == interval.attempt + 1)
    ]
    next_start = next_attempt['start_time_dt'].iloc[0] if not next_attempt.empty else pd.NaT
    mask = (tp_tools.thread_id == interval.thread_id) & (tp_tools.start_time_dt > start.end_time_dt)
    if pd.notna(next_start):
        mask &= tp_tools.start_time_dt < next_start
    tools = tp_tools[mask & tp_tools.name.isin(MUTATION_TOOLS)]
    counts = tools['name'].value_counts().to_dict()
    mutation_mix_rows.append({
        'travelplan_title': interval.travelplan_title,
        'attempt': interval.attempt,
        **{name: counts.get(name, 0) for name in ['add_slot', 'insert_slot', 'delete_slot', 'add_day', 'remove_day']},
    })
mutation_mix = pd.DataFrame(mutation_mix_rows)
display(mutation_mix)


## 5. Duplicate Repair Detection

This looks for repeated mutations against the same day/position or identical normalized tool calls inside a single repair interval. These are a strong trace-level signal of backtracking or repair churn.

In [ ]:
duplicate_rows = []
for _, interval in repair_intervals.iterrows():
    failed = validator_attempts[
        (validator_attempts.thread_id == interval.thread_id)
        & (validator_attempts.validation_attempt == interval.attempt)
        & (validator_attempts.validation_passed == False)
    ].iloc[0]
    next_attempt = validator_attempts[
        (validator_attempts.thread_id == interval.thread_id)
        & (validator_attempts.validation_attempt == interval.attempt + 1)
    ]
    next_start = next_attempt['start_time_dt'].iloc[0] if not next_attempt.empty else pd.NaT
    mask = (tp_tools.thread_id == interval.thread_id) & (tp_tools.start_time_dt > failed.end_time_dt)
    if pd.notna(next_start):
        mask &= tp_tools.start_time_dt < next_start
    tools = tp_tools[mask & tp_tools.name.isin(MUTATION_TOOLS)].copy()
    if tools.empty:
        continue
    tools['day_index'] = tools['tool_args'].map(lambda d: d.get('day_index') if isinstance(d, dict) else None)
    tools['position'] = tools['tool_args'].map(lambda d: d.get('position') if isinstance(d, dict) else None)
    by_position = (
        tools.groupby(['name', 'day_index', 'position'], dropna=False)
        .size()
        .reset_index(name='count')
        .query('count > 1')
    )
    by_args = (
        tools.groupby(['name', 'args_norm'], dropna=False)
        .size()
        .reset_index(name='count')
        .query('count > 1')
    )
    duplicate_rows.append({
        'travelplan_title': interval.travelplan_title,
        'attempt': interval.attempt,
        'repeated_same_day_position_groups': len(by_position),
        'repeated_exact_mutation_groups': len(by_args),
        'max_same_day_position_repeat': int(by_position['count'].max()) if not by_position.empty else 0,
        'max_exact_mutation_repeat': int(by_args['count'].max()) if not by_args.empty else 0,
    })

duplicate_repair_table = pd.DataFrame(duplicate_rows).sort_values(
    ['repeated_same_day_position_groups', 'repeated_exact_mutation_groups'],
    ascending=False,
)
display(duplicate_repair_table)


## 6. Cost Trajectory During Repairs

Cost is only available when the agent calls `cost_summary` or when the validator text states a total. This table tracks within-repair `cost_summary` calls.

In [ ]:
cost_rows = []
for _, interval in repair_intervals.iterrows():
    failed = validator_attempts[
        (validator_attempts.thread_id == interval.thread_id)
        & (validator_attempts.validation_attempt == interval.attempt)
        & (validator_attempts.validation_passed == False)
    ].iloc[0]
    next_attempt = validator_attempts[
        (validator_attempts.thread_id == interval.thread_id)
        & (validator_attempts.validation_attempt == interval.attempt + 1)
    ]
    next_start = next_attempt['start_time_dt'].iloc[0] if not next_attempt.empty else pd.NaT
    mask = (tp_tools.thread_id == interval.thread_id) & (tp_tools.start_time_dt > failed.end_time_dt)
    if pd.notna(next_start):
        mask &= tp_tools.start_time_dt < next_start
    costs = tp_tools[mask & (tp_tools.name == 'cost_summary')].sort_values('start_time_dt')
    for idx, row in enumerate(costs.itertuples(), start=1):
        cost_rows.append({
            'travelplan_title': interval.travelplan_title,
            'attempt': interval.attempt,
            'cost_summary_index': idx,
            'start_time': row.start_time,
            'total_cost': row.tool_cost_total,
            'content': row.output_text,
        })

cost_trajectory = pd.DataFrame(cost_rows)
display(cost_trajectory[['travelplan_title', 'attempt', 'cost_summary_index', 'total_cost', 'content']])

cost_summary_by_repair = (
    cost_trajectory.groupby(['travelplan_title', 'attempt'])
    .agg(first_total=('total_cost', 'first'), last_total=('total_cost', 'last'), calls=('total_cost', 'size'))
    .reset_index()
)
cost_summary_by_repair['delta'] = cost_summary_by_repair['last_total'] - cost_summary_by_repair['first_total']
display(cost_summary_by_repair)


## 7. Evidence Survival Analysis

This approximates whether uncertainty or fragile evidence survived into the final itinerary. It combines tool-output uncertainty signals with final local TravelPlan JSON artifacts.

In [ ]:
def load_local_travelplan(row: pd.Series) -> dict[str, Any] | None:
    path = LOCAL_TRACE_ROOT / row['local_file']
    obj = safe_loads(path.read_text(encoding='utf-8')) if path.exists() else None
    if isinstance(obj, dict):
        return (obj.get('outputs') or {}).get('travelplan')
    return None


def walk_slots(plan: dict[str, Any] | None):
    if not isinstance(plan, dict):
        return
    for day in plan.get('days') or []:
        for slot in day.get('slots') or []:
            yield day, slot


def slot_text(slot: dict[str, Any]) -> str:
    return ' '.join(str(slot.get(k) or '') for k in ['name', 'description', 'location', 'notes'])


def slot_links(slot: dict[str, Any]) -> list[str]:
    links = slot.get('links') or []
    return [str(link) for link in links]


evidence_rows = []
for _, m in tp_manifest.iterrows():
    plan = load_local_travelplan(m)
    slots = list(walk_slots(plan) or [])
    total_slots = len(slots)
    missing_links = 0
    fragile_links = 0
    uncertainty_slots = 0
    for _, slot in slots:
        links = slot_links(slot)
        if not links:
            missing_links += 1
        if any(domain in link for link in links for domain in FRAGILE_DOMAINS):
            fragile_links += 1
        if UNCERTAIN_RE.search(slot_text(slot)):
            uncertainty_slots += 1
    thread_tools = tp_tools[tp_tools.thread_id == m.thread_id]
    evidence_rows.append({
        'travelplan_title': m.travelplan_title,
        'thread_id': m.thread_id,
        'validation_attempts': m.validation_attempts,
        'total_slots': total_slots,
        'tool_error_outputs': int(thread_tools['has_error_language'].sum()),
        'tool_uncertainty_outputs': int(thread_tools['has_uncertainty_language'].sum()),
        'final_slots_missing_links': missing_links,
        'final_slots_fragile_links': fragile_links,
        'final_slots_with_uncertainty_language': uncertainty_slots,
    })

evidence_survival = pd.DataFrame(evidence_rows).sort_values(
    ['validation_attempts', 'final_slots_with_uncertainty_language', 'final_slots_missing_links'],
    ascending=False,
)
display(evidence_survival)


## 8. Baseline vs. Travel Agent Trace Shape

The baseline mostly performs explicit Tavily search calls. Travel Agent has a broader trace shape: search, route/cost checks, state reads, and TravelPlan mutation tools.

In [ ]:
def tool_group(name: str) -> str:
    if name in SEARCH_TOOLS:
        return 'Search tools'
    if name in MUTATION_TOOLS:
        return 'TravelPlan mutation tools'
    if name in READONLY_CHECK_TOOLS:
        return 'Read/check tools'
    if name == 'write_todos':
        return 'State-management tools'
    return 'Other'

travel_shape = (
    tp_tools.assign(system='travel_agent', tool_group=tp_tools['name'].map(tool_group))
    .groupby(['system', 'tool_group'])
    .size()
    .reset_index(name='tool_calls')
)
base_shape = (
    bl_tools.assign(system='baseline', tool_group='Search tools')
    .groupby(['system', 'tool_group'])
    .size()
    .reset_index(name='tool_calls')
)
trace_shape = pd.concat([travel_shape, base_shape], ignore_index=True)
display(trace_shape.sort_values(['system', 'tool_calls'], ascending=[True, False]))

trace_shape_pivot = trace_shape.pivot(index='tool_group', columns='system', values='tool_calls').fillna(0).astype(int).reset_index()
display(trace_shape_pivot)


## 9. Presentation-Ready Repair Flow Table

This table is designed to be copied into slides. It links validator issue type, repair tools, mutation breadth, and the next validator outcome.

In [ ]:
def summarize_tool_counts(counts: dict[str, int]) -> str:
    if not isinstance(counts, dict):
        return ''
    important = ['search_flights', 'search_hotels', 'search_web', 'search_restaurants', 'check_route_timing', 'delete_slot', 'insert_slot', 'add_slot', 'cost_summary']
    parts = [f'{name}={counts[name]}' for name in important if counts.get(name, 0)]
    return ', '.join(parts)

repair_flow_table = repair_intervals.copy()
repair_flow_table['tools_used'] = repair_flow_table['tool_counts'].map(summarize_tool_counts)
repair_flow_table['validator_requested'] = repair_flow_table['feedback_preview']
repair_flow_table['repair_scope'] = repair_flow_table.apply(
    lambda r: f"{int(r['mutation_calls'])} mutations across {int(r['distinct_days_mutated'])} day(s)",
    axis=1,
)
repair_flow_table = repair_flow_table[[
    'travelplan_title', 'attempt', 'issue_types', 'validator_requested',
    'tools_used', 'repair_scope', 'next_validation_passed'
]].sort_values(['travelplan_title', 'attempt'])
display(repair_flow_table)


## 10. Main Findings

- Validator feedback was actionable: every failed validator attempt was followed by targeted searches, route/cost checks, and TravelPlan mutation tools.
- The strongest repair signal is mutation churn, not repeated search. Tokyo attempt 2 and Marrakech attempt 1 rewrote many slots after one validator rejection.
- Some repairs were broad relative to the complaint. For example, a flight or timing issue often led to edits across multiple days.
- Cost repair is visible through repeated `cost_summary` calls, but cost checks are sparse and happen after batches of edits rather than after every budget-relevant slot.
- Evidence uncertainty can survive into final plans as missing links, fragile domains, or uncertainty language in slot descriptions/notes.
- Baseline traces are shallow and mostly search-only; Travel Agent traces are richer but expose repair-loop failure modes.